# Day 3 — Reranking & Query Expansion

---

Your RAG works. But sometimes the "top 3" chunks it retrieves aren't the *actually* best ones — they just look similar to the query.

Today we make retrieval smarter with two techniques used in every serious 2026 RAG system:

1. **Reranking** — pull more candidates, then re-score them with a smarter (slower) model
2. **Query expansion (HyDE-lite)** — rewrite the user's question before searching


## 1. The retrieval bottleneck

The bi-encoder embeddings from Section 5 are **fast but rough**. They map every doc to one vector and compare with cosine. That means:

- They can miss **subtle relevance** — a doc that partially matches
- They can rank a **word-similar-but-topic-different** doc above a truly relevant one

The fix is a **two-stage retrieval pipeline**:

```
    query
      │
      ▼
  ┌───────────┐
  │ Embedding │  fast, retrieves 20-50 candidates
  │  search   │
  └───────────┘
      │
      ▼
  ┌───────────┐
  │ Reranker  │  slow, re-scores the 20-50 more carefully
  └───────────┘
      │
      ▼
   top 3-5   ← what goes into the prompt
```

You get the **speed of embeddings** AND the **accuracy of a careful model**.


## 2. What's a cross-encoder reranker?


- **Cross-encoder**: takes `(query, doc)` **together** and outputs a relevance score. Much more accurate. Much slower.

Cross-encoders are too slow to use on your whole knowledge base — but perfect for scoring the top 20 candidates from your embedding search.

**The go-to open-source reranker: `BAAI/bge-reranker-base`** (free, runs on CPU).


In [ ]:
!pip install sentence-transformers chromadb --quiet

In [13]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-base")

# Test it on 4 candidate docs
query = "How do I reset my AcmeCloud password?"
candidates = [
    "AcmeCloud servers are in AWS us-east-1.",
    "To reset your password, click 'Forgot Password' on the login page.",
    "The Pro plan costs $29/month.",
    "Enterprise customers get a dedicated account manager.",
]

pairs = [(query, c) for c in candidates]
scores = reranker.predict(pairs)

for c, s in sorted(zip(candidates, scores), key=lambda x: -x[1]):
    print(f"  {s:+.3f}  {c}")


  +0.658  To reset your password, click 'Forgot Password' on the login page.
  +0.000  AcmeCloud servers are in AWS us-east-1.
  +0.000  The Pro plan costs $29/month.
  +0.000  Enterprise customers get a dedicated account manager.


The reset-password doc gets a **much higher** score than the others — even though other docs might have been closer in embedding space if they shared vocabulary. That's the reranker doing its job.


## 3. Two-stage retrieval in practice


In [14]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
kb = client.get_or_create_collection("acme_v2")

docs = [
    "AcmeCloud's free tier includes 10 GB of storage.",
    "The Pro plan costs $29/month and includes 500 GB.",
    "AcmeCloud servers are located in AWS us-east-1 and eu-west-1.",
    "To reset your password, click 'Forgot Password' on the login page.",
    "Enterprise customers receive 24/7 phone support.",
    "AcmeCloud was founded in 2019 by Priya Rao and Marcus Chen.",
    "Passwords must be at least 12 characters and include a symbol.",
    "You can enable two-factor authentication in the security settings.",
]
kb.add(
    documents=docs,
    embeddings=model.encode(docs).tolist(),
    ids=[f"d{i}" for i in range(len(docs))],
)

def retrieve_and_rerank(question: str, k_candidates: int = 6, k_final: int = 3):
    # Stage 1: fast embedding search
    q_vec = model.encode([question]).tolist()
    r = kb.query(query_embeddings=q_vec, n_results=k_candidates)
    candidates = r["documents"][0]

    # Stage 2: rerank with cross-encoder
    scores = reranker.predict([(question, c) for c in candidates])
    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    return [doc for doc, _ in ranked[:k_final]]

for doc in retrieve_and_rerank("how do I change my password?"):
    print(" -", doc)


 - To reset your password, click 'Forgot Password' on the login page.
 - Passwords must be at least 12 characters and include a symbol.
 - You can enable two-factor authentication in the security settings.


**What to notice:** you asked about "change" but the doc says "reset." Embedding search picks up on the semantic overlap; the reranker confirms the ordering. A pure-embedding search *might* have surfaced the "12 characters and a symbol" doc above the reset doc. Reranking prevents that.


## 4. Query expansion — the "HyDE-lite" trick

Sometimes users ask questions in ways that don't match how the document is written:

- User: `"is it safe?"`
- Doc: `"AcmeCloud uses AES-256 encryption at rest and TLS 1.3 in transit."`

The embedding of `"is it safe?"` isn't very close to the encryption doc — too generic.

**HyDE (Hypothetical Document Embeddings)** solves this: ask the LLM to *write a fake answer* to the question, then embed *that* and use it for search. The fake answer looks more like a real doc, so it retrieves better.

We'll use a **light version**: just prepend a rewritten query. Same idea, less complexity.


In [17]:
from together import Together
from dotenv import load_dotenv
load_dotenv()

llm = Together()

def expand_query(question: str) -> str:
    prompt = f"""You are rewriting a user question for a company documentation search.
Use likely product-domain terms, not generic filler words.
Include synonyms and exact security/privacy terms that appear in docs, such as:
security, encryption, AES-256, TLS, data protection, privacy, secure, safe,
authentication, compliance, access control.
Return only a short keyword query with 6-12 terms, no sentence, no explanation.

User question: {question}

Rewritten query:"""
    resp = llm.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=8000,
    )
    return resp.choices[0].message.content.strip()

original = "is it safe?"
expanded = expand_query(original)
print(f"Original: {original}")
print(f"Expanded: {expanded}")


Original: is it safe?
Expanded: security encryption AES-256 TLS data protection privacy secure safe authentication compliance access control


You should see something like `"security encryption safety AcmeCloud data protection TLS AES"`. Now embed the *expanded* string and search — you'll get much better results.


In [18]:
def rag_with_expansion(question: str, k_final: int = 3):
    expanded = expand_query(question)
    # Use the EXPANDED string for retrieval, but the ORIGINAL for the answer prompt
    hits = retrieve_and_rerank(expanded, k_candidates=6, k_final=k_final)
    return {"expanded_query": expanded, "chunks": hits}

r = rag_with_expansion("how do I stay secure?")
print("Expanded to:", r["expanded_query"])
for c in r["chunks"]:
    print(" -", c)


Expanded to: security encryption AES-256 TLS data protection privacy authentication compliance access control safe
 - You can enable two-factor authentication in the security settings.
 - Passwords must be at least 12 characters and include a symbol.
 - AcmeCloud servers are located in AWS us-east-1 and eu-west-1.


## 5. When each helps

| Technique | When it helps most | When it doesn't |
|---|---|---|
| **Reranking** | Always. Nearly-free quality boost when you can afford ~50 ms extra. | Only when embedding-search top-k already covers the right doc. |
| **Query expansion** | Vague or under-specified user questions. | Long, keyword-rich questions (no need). |

**Rule for freshers:** turn on **reranking by default**. Add **query expansion** only if you see users typing short, vague questions.


## Recap

- Two-stage retrieval: **cheap embedding search** → **expensive reranker**. Better than either alone.
- Use `BAAI/bge-reranker-base` — free, open-source, runs on CPU.
- **Query expansion** rewrites vague user questions into keyword-rich search strings.
- Cohere Rerank is the paid API alternative — same idea, better numbers, costs money.
- **Next class:** designing the RAG prompt and managing context-window limits.
